In [4]:
import polars as pl
from rapidfuzz import fuzz

In [5]:
answers = pl.read_parquet("../data_sampled/cases_answer_key.parquet")
corrupted = pl.read_parquet("../data_sampled/corrupted_cases_labeled.parquet")
clean = pl.read_parquet("../data_sampled/cases_sampled.parquet")

In [6]:
clean.lazy().collect_schema()

Schema([('ddl_case_id', String),
        ('year', Int64),
        ('state_code', Int64),
        ('dist_code', Int64),
        ('court_no', Int64),
        ('cino', String),
        ('judge_position', String),
        ('female_defendant', String),
        ('female_petitioner', String),
        ('female_adv_def', Int64),
        ('female_adv_pet', Int64),
        ('type_name', Int64),
        ('purpose_name', Int64),
        ('disp_name', Int64),
        ('date_of_filing', Date),
        ('date_of_decision', Date),
        ('date_first_list', Date),
        ('date_last_list', Date),
        ('date_next_list', Date),
        ('type_name_s', String),
        ('disp_name_s', String),
        ('purpose_name_s', String),
        ('state_name', String),
        ('district_name', String),
        ('court_name', String),
        ('ddl_filing_judge_id', Int64),
        ('ddl_decision_judge_id', Int64),
        ('female_judge_filing', String),
        ('start_date_filing', Date),
        ('end_da

In [7]:
corrupted.lazy().collect_schema()

Schema([('record_id', UInt32),
        ('year', Int64),
        ('judge_position', String),
        ('female_defendant', String),
        ('female_petitioner', String),
        ('female_adv_def', Int64),
        ('female_adv_pet', Int64),
        ('date_of_filing', Date),
        ('date_of_decision', Date),
        ('type_name_s', String),
        ('disp_name_s', String),
        ('purpose_name_s', String),
        ('state_name', String),
        ('district_name', String),
        ('court_name', String),
        ('female_judge_filing', String),
        ('female_judge_decision', String),
        ('act_s', List(String)),
        ('section_s', List(String)),
        ('number_sections_ipc', Int64),
        ('criminal', Int64),
        ('bailable_ipc', String),
        ('n_acts_or_sections', UInt32)])

In [8]:
clean = clean.select(["ddl_case_id",
        'year',
        'judge_position',
        'female_defendant',
        'female_petitioner',
        'female_adv_def',
        'female_adv_pet',
        'date_of_filing',
        'date_of_decision', 
        'type_name_s', 
        'disp_name_s', 
        'purpose_name_s', 
        'state_name', 
        'district_name', 
        'court_name', 
        'female_judge_filing', 
        'female_judge_decision', 
        'act_s', 
        'section_s', 
        'number_sections_ipc', 
        'criminal', 
        'bailable_ipc', 
        'n_acts_or_sections'])

In [9]:
print(corrupted.shape)
print(clean.shape)
print(answers.shape)

(50000, 23)
(50000, 23)
(50000, 2)


In [10]:
join = []
for c in corrupted.columns:
    if c not in ["record_id", "act_s", "section_s"]:
        join.append(c)

In [11]:
matches = corrupted.join(clean, on=join, how="inner")
print(matches.height)

4829


In [12]:
matches = corrupted.join(clean, on=join, how="inner", nulls_equal=True)
print(matches.height)

27887


In [13]:
matches.select(pl.col("record_id").n_unique())

record_id
u32
27270


In [14]:
matches_joined = matches.join(answers, on="record_id", how="left", suffix="_true")
pairs = matches_joined.filter(pl.col("ddl_case_id") == pl.col("ddl_case_id_true"))
print(pairs.height)

27255


In [15]:
precision = pairs.height / 27_887
print(f"precision percent of {precision * 100} percent via exact column-column matching.")
correctly_found = pairs.height / 50_000
print(f"recall percent of {correctly_found * 100} percent of original via exactly column-column matching.")

precision percent of 97.73371104815864 percent via exact column-column matching.
recall percent of 54.510000000000005 percent of original via exactly column-column matching.


In [16]:
clean.group_by("state_name").agg(pl.len()).with_columns((pl.col("len") * pl.col("len")).alias("pairings")).select(pl.col("pairings").sum())

pairings
u32
204713292


In [17]:
per_state = clean.group_by("state_name").agg(pl.len()).with_columns((pl.col("len") * pl.col("len")).alias("pairings"))

per_state.sort("pairings", descending=True)

state_name,len,pairings
str,u32,u32
"""Uttar Pradesh""",9847,96963409
"""Bihar""",4532,20539024
"""Maharashtra""",4250,18062500
"""Madhya Pradesh""",3390,11492100
"""Rajasthan""",3383,11444689
…,…,…
"""Sikkim""",20,400
"""Meghalaya""",7,49
"""DNH at Silvasa""",6,36


In [18]:
print(50_000 * 50_000)
print((50_000 * 50_000) / 204713292)

2500000000
12.212201638572644


In [19]:
per_state_and_judgep = clean.group_by(["state_name", "judge_position"]).agg(pl.len()).with_columns((pl.col("len") * pl.col("len")).alias("pairings"))

In [20]:
per_state_and_judgep.sort("pairings", descending = True)

state_name,judge_position,len,pairings
str,str,u32,u32
"""Uttar Pradesh""","""chief judicial magistrate""",4774,22791076
"""Uttar Pradesh""","""district and sessions court""",2255,5085025
"""Bihar""","""judicial magistrate court""",2177,4739329
"""Madhya Pradesh""","""district and sessions court""",1947,3790809
"""Bihar""","""district and sessions court""",1729,2989441
…,…,…,…
"""Orissa""","""a.d.j.,salipur""",1,1
"""Telangana""","""2-additional district and sess…",1,1
"""Meghalaya""","""additional deputy commissioner""",1,1


In [21]:
per_state_and_judgep.select(pl.col("pairings").sum())

pairings
u32
60203136


In [22]:
per_state_and_judge_and_outcome = clean.group_by(["state_name", "judge_position", "disp_name_s"]).agg(pl.len()).with_columns((pl.col("len") * pl.col("len")).alias("pairings"))

In [23]:
per_state_and_judge_and_outcome.select(pl.col("pairings").sum())

pairings
u32
19629930


In [24]:
clean_trimmed = clean.select(["ddl_case_id",
                              "state_name",
                              "judge_position",
                              "disp_name_s",
                              "date_of_filing",
                              "date_of_decision",
                              "district_name",
                              "court_name",
                              "type_name_s",
                              "female_defendant",
                              "female_petitioner"
                                ])

corrupted_trimmed = corrupted.select(["record_id",
                              "state_name",
                              "judge_position",
                              "disp_name_s",
                              "date_of_filing",
                              "date_of_decision",
                              "district_name",
                              "court_name",
                              "type_name_s",
                              "female_defendant",
                              "female_petitioner"
                                ])

In [25]:
both_joined = corrupted_trimmed.join(clean_trimmed, on=["state_name", "judge_position", "disp_name_s"], how="inner", suffix="_clean")
both_joined.height

19629930

In [26]:
check_matches = both_joined.join(answers, on="record_id", how="left", suffix="_true")
matches = check_matches.filter(pl.col("ddl_case_id") == pl.col("ddl_case_id_true"))
print(matches.height)

50000


In [27]:
scored = both_joined.with_columns((pl.col("date_of_filing") - pl.col("date_of_filing_clean")).dt.total_days().abs().alias("filing_diff"))
scored = scored.with_columns((pl.col("date_of_decision") - pl.col("date_of_decision_clean")).dt.total_days().abs().alias("decision_diff"))

scored = scored.with_columns(
    (pl.col("type_name_s") == pl.col("type_name_s_clean")).alias("type_name_s_agree"),
    (pl.col("female_defendant") == pl.col("female_defendant_clean")).alias("female_defendant_agree"),
    (pl.col("female_petitioner") == pl.col("female_petitioner_clean")).alias("female_petitioner_agree"),
)

In [28]:
from rapidfuzz import process

In [30]:
district_sim = process.cpdist(
    scored["district_name"].fill_null(""),
    scored["district_name_clean"].fill_null(""),
    scorer=fuzz.ratio
)

court_sim = process.cpdist(
    scored["court_name"].fill_null(""),
    scored["court_name_clean"].fill_null(""),
    scorer=fuzz.ratio
)

In [31]:
scored = scored.with_columns(pl.Series("district_sim", district_sim), pl.Series("court_sim", court_sim))

In [32]:
scored.head()

record_id,state_name,judge_position,disp_name_s,date_of_filing,date_of_decision,district_name,court_name,type_name_s,female_defendant,female_petitioner,ddl_case_id,date_of_filing_clean,date_of_decision_clean,district_name_clean,court_name_clean,type_name_s_clean,female_defendant_clean,female_petitioner_clean,filing_diff,decision_diff,type_name_s_agree,female_defendant_agree,female_petitioner_agree,district_sim,court_sim
u32,str,str,str,date,date,str,str,str,str,str,str,date,date,str,str,str,str,str,i64,i64,bool,bool,bool,f32,f32
243,"""Maharashtra""","""chief judicial magistrate""","""disposition var missing""",2018-08-13,null,"""Osmanabad""","""CHIEF JUDICIAL MAGISTRATE COUR…","""s.c.c.""","""0 male""","""-9998 unclear""","""01-01-01-201900001082018""",2018-03-20,null,"""Nandurbar""","""Chief Judcial Magistrate Nandu…","""cri.m.a.""","""0 male""","""1 female""",146,null,false,true,false,44.444443,18.421053
292,"""Maharashtra""","""chief judicial magistrate""","""disposition var missing""",2018-10-26,null,"""Chandrapur""","""Chief Judicial Magistrate, Cha…","""s.c.c.""","""0 male""",null,"""01-01-01-201900001082018""",2018-03-20,null,"""Nandurbar""","""Chief Judcial Magistrate Nandu…","""cri.m.a.""","""0 male""","""1 female""",220,null,false,true,null,63.157894,87.323944
388,"""Maharashtra""","""chief judicial magistrate""","""disposition var missing""",2018-11-29,null,"""Beed""","""CHIEF JUDICIAL MAGISTRATE BEED…","""juvenile""","""1 female""","""-9998 unclear""","""01-01-01-201900001082018""",2018-03-20,null,"""Nandurbar""","""Chief Judcial Magistrate Nandu…","""cri.m.a.""","""0 male""","""1 female""",254,null,false,false,false,15.384615,18.461538
510,"""Maharashtra""","""chief judicial magistrate""","""disposition var missing""",2018-07-16,null,"""Kolhapur""","""Chief Judicial Magistrate Kolh…","""s.c.c.""","""1 female""","""-9998 unclear""","""01-01-01-201900001082018""",2018-03-20,null,"""Nandurbar""","""Chief Judcial Magistrate Nandu…","""cri.m.a.""","""0 male""","""1 female""",118,null,false,false,false,35.294117,82.352943
526,"""Maharashtra""","""chief judicial magistrate""","""disposition var missing""",2018-12-04,null,"""Aurangabad""","""CHIEF JUDICIAL MAGISTRATE,AURA…","""r.c.c.""","""0 male""","""-9998 unclear""","""01-01-01-201900001082018""",2018-03-20,null,"""Nandurbar""","""Chief Judcial Magistrate Nandu…","""cri.m.a.""","""0 male""","""1 female""",259,null,false,true,false,42.105263,17.142857
